# Optimizer comparison

This is a generattor notbeook that runs the search and saves per-run results to CSV.
`03_results_and_significance.ipynb` reads the saved file and performs the analysis.
Re-running this notebook is only needed to add seeds or datasets.

**Method**

- **Data**: 20 datasets in `datasets.DATASETS`
- **Splits**: ten seeded 60/20/20 train/validation/test splits per dataset
  (`methodology.three_way_split`)
- **Per-feature kappa**: every method searches an `n_features`-dimensional kappa, one value per
  feature column, plus the single eta dimension. `datasets.INIT_STRATEGIES`' scalar
  `(kappa0, eta0)` starts are broadcast to a per-feature vector at the point of use
  (`methodology.run_tuning_search`), and the grid-search baseline searches the same
  `(n_features + 1)`-dimensional space directly
  (`methodology.run_grid_search_baseline`). No method is given a richer model than another.
- **Gradient methods**: Adam, our L-BFGS, and SciPy's L-BFGS-B, each tuned over
  `datasets.OPT_GRID`'s three configurations by `datasets.INIT_STRATEGIES`'.(`methodology.run_tuning_search`).
- **Baseline**: grid search, a derivative-free column-incremental exhaustive search over
  `(kappa_1, ..., kappa_p, eta)` space (`methodology.run_grid_search_baseline`). The resolution is set by the `SEARCH_BUDGET`.
- **t_max budget**: for every `(dataset, seed)` pair, grid search runs first and its
  measured wall-clock cost becomes `t_max`, the total search budget shared by the
  three gradient methods Because `t_max` is measured fresh per group rather than fixed once, a gradient
  method's total tuning time can never exceed the grid search it is compared against on that
  same dataset and seed: it can match it, if no run converges before its runs out, or
  beat it, if runs converge early.
- **Test evaluation**: the winning `(kappa*, eta*)` from each group is forward-evaluated on the
  test split once (`methodology.evaluate_test_once`).

`datasets.DATASETS` loads Wine Quality at its full 6497 rows (not a subsample); its entry in
`SEARCH_BUDGET_OVERRIDES` below drops its grid resolution to 256 rather than the panel's 2048,
since grid search's cost scales with `N^2` and Wine Quality is far larger than every other
dataset here.


In [ ]:
import numpy as np
import pandas as pd

import methodology as meth
from datasets import DATASETS, INIT_STRATEGIES, N_CFGS_PER_OPTIMIZER, OPT_GRID

SEEDS = np.arange(0, 10, 1)
BOUNDS = (1.0, 80.0)            # kappa_bounds == eta_bounds
MAX_ITERS = 200                 # exercises every code path; not a convergence guarantee
TOL = 1e-6                      # relative loss-change tolerance, matches scipy L-BFGS-B's ftol formula
SEARCH_BUDGET = 2048            # Grid search baseline resolution for the panel

SEARCH_BUDGET_OVERRIDES = {"UCI: Wine Quality": 256}   # Wine Quality loads at its full 6497 rows; grid search cost scales with N^2, so it gets a coarser resolution
assert set(SEARCH_BUDGET_OVERRIDES) <= set(DATASETS), "override keys must name real datasets"

N_INITS = len(INIT_STRATEGIES)
assert N_INITS == 5
assert N_CFGS_PER_OPTIMIZER == 3

OPTIMIZERS = {
    "Adam": (meth.run_adam, OPT_GRID["Adam"]),
    "L-BFGS": (meth.run_lbfgs, OPT_GRID["L-BFGS"]),
    "L-BFGS-B": (meth.run_lbfgsb, OPT_GRID["L-BFGS-B"]),
}
assert set(OPTIMIZERS) == set(OPT_GRID), "every optimizer must draw its configs from OPT_GRID"


## Main experiment loop

In [ ]:
results = []
all_test_reads = []
t_max_values = []
n_features_by_dataset = {}
n_samples_by_dataset = {}
budget_by_dataset = {}

for dataset_name, dataset_fn in DATASETS.items():
    X, y = dataset_fn()
    n_features = X.shape[1]
    n_samples = X.shape[0]
    n_features_by_dataset[dataset_name] = n_features
    n_samples_by_dataset[dataset_name] = n_samples

    # Grid resolution for this dataset: the panel default unless overridden.
    budget = SEARCH_BUDGET_OVERRIDES.get(dataset_name, SEARCH_BUDGET)
    budget_by_dataset[dataset_name] = budget

    for seed in SEEDS:
        X_train, y_train, X_val, y_val, X_test, y_test = meth.three_way_split(X, y, seed)
        baseline = meth.run_grid_search_baseline(X_train, y_train, X_val, y_val, BOUNDS, budget)
        t_max = baseline["wall_clock"]
        t_max_values.append(t_max)

        baseline_label = f"{dataset_name}/Grid search/seed{seed}"
        baseline_test_mse = meth.evaluate_test_once(
            all_test_reads, baseline_label, X_test, X_train, y_train, y_test,
            baseline["kappa"], baseline["eta"],
        )
        results.append(dict(
            dataset=dataset_name, optimizer="Grid search", seed=seed, init=None, cfg=None,
            val_mse=baseline["val_mse"], test_mse=baseline_test_mse,
            nfev=baseline["n_evals"], njev=0, n_linesearch=0, nit=None,
            wall_clock=baseline["wall_clock"], stop_reason=None, t_max=t_max,
            kappa=str(np.atleast_1d(baseline["kappa"]).tolist()), eta=float(baseline["eta"]),
            search_budget=budget, n_samples=n_samples,
        ))

        for opt_name, (run_fn, cfg_grid) in OPTIMIZERS.items():
            grid_runs, (best_cfg, best_init, best_result) = meth.run_tuning_search(
                run_fn, cfg_grid, INIT_STRATEGIES, X_train, y_train, X_val, y_val,
                t_max=t_max, max_iters=MAX_ITERS, tol=TOL, bounds=BOUNDS, n_features=n_features,
            )

            # Test is read exactly once per (dataset, seed, optimizer): a single
            # forward-pass prediction at the (kappa*, eta*) selected entirely on
            # validation.
            label = f"{dataset_name}/{opt_name}/seed{seed}"
            test_mse = meth.evaluate_test_once(
                all_test_reads, label, X_test, X_train, y_train, y_test,
                best_result["kappa"], best_result["eta"],
            )

            for cfg, init_name, r in grid_runs:
                results.append(dict(
                    dataset=dataset_name, optimizer=opt_name, seed=seed, init=init_name, cfg=str(cfg),
                    val_mse=r["val_mse"], test_mse=(test_mse if r is best_result else np.nan),
                    nfev=r["nfev"], njev=r["njev"], n_linesearch=r["n_linesearch"], nit=r["nit"],
                    wall_clock=r["wall_clock"], stop_reason=r["stop_reason"], t_max=t_max,
                    kappa=str(np.atleast_1d(r["kappa"]).tolist()), eta=float(r["eta"]),
                    search_budget=budget, n_samples=n_samples,
                ))

df = pd.DataFrame(results)
print(
    f"{len(df)} rows: {len(DATASETS)} datasets x {len(SEEDS)} seeds x "
    f"({len(OPTIMIZERS)} gradient methods x {N_CFGS_PER_OPTIMIZER} cfgs x {N_INITS} inits + 1 baseline)"
)
print(
    f"t_max per group: min={min(t_max_values):.4f}s, "
    f"mean={sum(t_max_values) / len(t_max_values):.4f}s, max={max(t_max_values):.4f}s"
)
print("grid resolution actually used per dataset:")
for name, b in budget_by_dataset.items():
    tag = "  <-- override" if name in SEARCH_BUDGET_OVERRIDES else ""
    print(f"  {name:<28} N={n_samples_by_dataset[name]:>5}  budget={b}{tag}")
df.head()


## Verification

In [ ]:
expected_test_reads = len(DATASETS) * len(SEEDS) * (len(OPTIMIZERS) + 1)
expected_result_rows = len(DATASETS) * len(SEEDS) * (len(OPTIMIZERS) * N_CFGS_PER_OPTIMIZER * N_INITS + 1)
assert len(all_test_reads) == expected_test_reads, f"expected {expected_test_reads} test reads, got {len(all_test_reads)}"
assert len(set(all_test_reads)) == expected_test_reads, "duplicate test reads detected"
assert len(df) == expected_result_rows, f"expected {expected_result_rows} result rows, got {len(df)}"

gradient_rows_check = df[df["optimizer"].isin(OPTIMIZERS.keys())]
non_null_counts = gradient_rows_check.groupby(["dataset", "optimizer", "seed"])["test_mse"].apply(lambda s: s.notna().sum())
assert (non_null_counts == 1).all(), "each (dataset, optimizer, seed) group must have exactly one test_mse"
idxmin_rows = gradient_rows_check.loc[gradient_rows_check.groupby(["dataset", "optimizer", "seed"])["val_mse"].idxmin()]
assert idxmin_rows["test_mse"].notna().all(), "the val_mse-argmin row must be the one with the logged test_mse"

print(f"Leakage guard OK: {len(all_test_reads)} test reads, all unique.")
print(f"Result rows OK: {len(df)} rows, exactly one test_mse per gradient-method group.")

# t_max guard: each gradient method's total search time for a group can never
# exceed that group's own t_max (Grid search's measured cost there). A run's
# internal loop only checks elapsed time once per iteration, so a small
# positive overage (one iteration's cost) is expected, not a violation.
per_group_total = (
    df[df["optimizer"].isin(OPTIMIZERS.keys())]
    .groupby(["dataset", "optimizer", "seed"])
    .agg(total_wall_clock=("wall_clock", "sum"), t_max=("t_max", "first"))
)
overage = per_group_total["total_wall_clock"] - per_group_total["t_max"]
print(f"t_max guard: max overage across all groups = {overage.max():.4f}s (should be small -- one iteration's cost, at most).")

# Per-feature kappa guard: every method (all four optimizers, including the
# Grid search baseline) must have searched a kappa vector whose length equals
# that dataset's own feature count -- not a scalar broadcast to every column.
import ast

df["kappa_dim"] = df["kappa"].apply(lambda s: len(ast.literal_eval(s)))
df["expected_kappa_dim"] = df["dataset"].map(n_features_by_dataset)
mismatches = df[df["kappa_dim"] != df["expected_kappa_dim"]]
assert mismatches.empty, f"found rows where kappa_dim != n_features:\n{mismatches[['dataset', 'optimizer', 'kappa_dim', 'expected_kappa_dim']]}"
per_dataset_optimizer_dim = df.groupby(["dataset", "optimizer"])["kappa_dim"].nunique()
assert (per_dataset_optimizer_dim == 1).all(), "kappa_dim is not constant within a (dataset, optimizer) group"
print("Per-feature kappa guard OK: every (dataset, optimizer) group's kappa_dim matches that dataset's n_features.")

# Budget/size guard. Every row must carry the grid resolution and row count its
# dataset was actually configured with, and each Grid search block's nfev must
# land just under its own budget (the staircase traversal spends nearly all of
# it). This catches a differently-parameterised run being upserted here under a
# colliding dataset label with a mismatched nfev, N, or t_max.
grid_rows = df[df["optimizer"] == "Grid search"]
for name, block in df.groupby("dataset"):
    configured_budget = budget_by_dataset[name]
    configured_n = n_samples_by_dataset[name]
    assert (block["search_budget"] == configured_budget).all(), \
        f"{name}: search_budget column disagrees with the configured budget {configured_budget}"
    assert (block["n_samples"] == configured_n).all(), \
        f"{name}: n_samples column disagrees with the loaded dataset size {configured_n}"
for name, block in grid_rows.groupby("dataset"):
    configured_budget = budget_by_dataset[name]
    lo, hi = 0.95 * configured_budget, configured_budget
    assert block["nfev"].between(lo, hi).all(), (
        f"{name}: Grid search nfev {sorted(block['nfev'].unique())} outside "
        f"[{lo:.0f}, {hi:.0f}] for a budget of {configured_budget} -- these rows were not "
        f"produced by this notebook's configuration"
    )
print("Budget/size guard OK: every dataset's rows carry its configured grid budget and row count.")
print(grid_rows.groupby("dataset").agg(N=("n_samples", "first"), budget=("search_budget", "first"),
                                       nfev=("nfev", "mean"), t_max=("t_max", "mean")).to_string())


## Save results

In [ ]:
RESULTS_PATH = (
    f"results/results-tmax-gridmatched-perfeature-cfg{N_CFGS_PER_OPTIMIZER}-init{N_INITS}"
    f"-ds{len(DATASETS)}-grid{SEARCH_BUDGET}.csv"
)
UPSERT_KEY = ["dataset", "optimizer", "seed", "init", "cfg"]

combined, n_replaced = meth.save_upsert(df, RESULTS_PATH, UPSERT_KEY)
print(f"Saved {len(combined)} total rows to {RESULTS_PATH}")
print(f"  this run: {len(df)} rows ({n_replaced} replaced existing rows, {len(df) - n_replaced} newly added)")
print("  seeds now covered per (dataset, optimizer):")
print(combined.groupby(["dataset", "optimizer"])["seed"].nunique().unstack("optimizer").to_string())
